In [11]:
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## MCPs

In [13]:
from langchain_mcp_adapters.client import MultiServerMCPClient

flights_client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

venues_client = MultiServerMCPClient(
    {
        "wedstay": {
                "transport": "streamable_http",
                "url": "https://www.thewedstay.com/api/ai/mcp"
            }
    }
)

playlist_client = MultiServerMCPClient({
    "musicbrainz": {
        "transport": "streamable_http",
        "url": "https://musicbrainz.caseyjhand.com/mcp",
    }
})


flight_tools = await flights_client.get_tools()
venue_tools = await venues_client.get_tools()
playlist_tools = await playlist_client.get_tools()

In [14]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

subagent_flight = create_agent(
    "gpt-5-nano",
    tools=flight_tools,
    system_prompt="You are a travel agent. No follow up questions."
)

subagent_venue = create_agent(
    "gpt-5-nano",
    tools=venue_tools,
    system_prompt="You are a venue agent. No follow up questions."
)

subagent_playlist = create_agent(
    "gpt-5-nano",
    tools=playlist_tools,
    system_prompt="You are a DJ artist. No follow up questions."
)

In [15]:
from langchain.messages import HumanMessage
from langchain.tools import tool

@tool
async def call_subagent_flight(source: str, destination: str, date: str) -> str:
    """Call subagent flight in order to find flights from source to the given destination"""
    response = await subagent_flight.ainvoke({"messages": [HumanMessage(content=f"Get me a direct flight from {source} to {destination} on {date}")]})
    return response["messages"][-1].content

@tool
async def call_subagent_venue(location: str, date: str, guestCount: int) -> str:
    """Call subagent venue in order to find venue in the given destination"""
    response = await subagent_venue.ainvoke({"messages": [HumanMessage(content=f"Get me a venue at {location} to host event for {guestCount} on {date}")]})
    return response["messages"][-1].content

@tool
async def call_subagent_playlist(genre: str) -> str:
    """Call subagent playlist in order to find songs playlist the given genre"""
    response = await subagent_playlist.ainvoke({"messages": [HumanMessage(content=f"Get me a playlist of 5 songs for genre {genre}")]})
    return response["messages"][-1].content

In [16]:
agent = create_agent(
    "gpt-5-nano",
    tools=[call_subagent_flight, call_subagent_venue, call_subagent_playlist],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a wedding coordinator. "
        "Ask for missing origin, destination, wedding date, guest count,and genre. "
        "Delegate flight, dj song playlist and venue searches to your specialist tools, "
        "then combine their results into a wedding plan."
    ),
)

In [17]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Plan user their wedding")]},
    config
)

In [20]:
print(response["messages"][-1].content)

I’d love to plan your wedding. To get started, I need a few key details:

- Origin: the city/country you’ll be flying from for guests (or your own origin if not traveling)
- Destination: the wedding location (city or venue you’re considering)
- Wedding date: exact date, or the date range if you’re flexible
- Guest count: approximate number of guests
- Genre/vibe for the wedding: the musical style or overall feel you want (e.g., classic romance, modern chic, rustic, beachy, indie, etc.)

Once I have these, I’ll run parallel searches for:
- Flights: from origin to destination for your guests
- Venues: suitable venues in the destination for your date and guest count
- Playlist/DJ: a playlist or DJ lineup aligned with your genre

If you have any preferences (budget, must-have venue features, travel considerations, dietary needs), feel free to share and I’ll incorporate them.


In [21]:
response = await agent.ainvoke(
    {
        "messages": [HumanMessage(content=(
            "Origin: London. "
            "Destination: Paris. "
            "Wedding date: 2027-06-20. "
            "Guest count: 100. "
            "Music genre: Jazz."
        ))]
    },
    config=config,
)

print(response["messages"][-1].content)

Wonderful — I’ve pulled together your initial plan based on London as the origin, Paris as the destination, 20 June 2027, 100 guests, and a Jazz vibe.

1) Quick plan snapshot
- Date: 2027-06-20
- Location: Paris (with a plan to host both ceremony and reception in or near Paris; open to nearby regions if needed)
- Guest count: ~100
- Vibe: Jazz-inspired elegance (classic but relaxed)

2) Guest travel options (London → Paris)
Here are solid flight options for guests, with price and schedule notes. Best price and a solid morning/afternoon option are highlighted.

- Best price: Luton (LTN) → Paris CDG, 06:05–08:30, 75 EUR
  Booking: https://kiwi.com/u/5349xj
- Shortest morning option: SEN → Paris CDG, 07:25–09:35, 96 EUR
  Booking: https://kiwi.com/u/9xqwmq
- Stansted (STN) → CDG, 13:00–15:20, 84 EUR
  Booking: https://kiwi.com/u/2qrph9
- Stansted (STN) → CDG, 19:00–21:20, 84–100 EUR range
  Booking: https://kiwi.com/u/ru5x3w
- Gatwick (LGW) → CDG, 18:00–20:20, 99–97 EUR
  Booking: https:/